# X-VLA F0/F1 trên LIBERO-Mem T1/T3, split V2 (Kaggle 2×T4)

Cùng pipeline với `kaggle_xvla_mem_finetune_f0_f1.ipynb` (bản đã sửa). Dữ liệu lấy thẳng từ shard V2 của SmolVLA:
ảnh, prompt theo từng frame và `robot_states` dùng lại nguyên vẹn; chỉ cột action được đổi delta 7D → absolute EEF-6D 10D
bằng công thức OSC_POSE của robosuite 1.4 (đã đối chiếu với simulator thật). Không cần simulator, không chép lại 12 GB ảnh.

**Input cần gắn (Add Input):**
1. `libero-mem-t1-t3-shards-v2` (shard V2, split lock `8275e468…`).
2. `xvla-v2-code` — dataset chứa thư mục `xvla_v2/` (các script mới trong `scripts/xvla_v2`).
3. Internet **ON** (clone X-VLA commit `6c4ecfb`, tải `2toINF/X-VLA-Libero` revision `129e714…`). Nếu tắt Internet, gắn thêm dataset chứa repo X-VLA và checkpoint.

**Thứ tự:** cài đặt → dựng dữ liệu → kiểm tra căn chỉnh với checkpoint → smoke 30 update → train F0 → train F1 → đóng gói.
Mỗi phiên Kaggle tối đa 12 giờ và `/kaggle/working` bị xóa khi mở phiên mới. Để chạy tiếp: Save Version (giữ output), ở phiên sau
**Add Input → Notebook output** của phiên trước; cell 2b chép `runs/` cũ về và các cell train tự resume từ `ckpt-*` mới nhất.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, hashlib

ARMS = ["F0", "F1"]          # bỏ bớt nếu chỉ muốn chạy một arm trong phiên này
ITERS = 5000                 # optimizer updates / arm (endpoint chính theo roadmap)
SAVE_EVERY = 500
GPUS = 2
BATCH_PER_GPU = 4            # micro-batch; nếu OOM ở smoke -> 2 và GRAD_ACCUM = 8
GRAD_ACCUM = 4               # hiệu dụng = BATCH_PER_GPU * GRAD_ACCUM * GPUS = 32
# Lịch LR cho X-VLA (configs/training/xvla_mem_adapter_v2.json), giống hệt cho F0 và F1:
LR, LR_COEF = 5e-5, 0.5      # transformer LoRA + action heads 5e-5; VLM LoRA + soft prompts 2.5e-5
FREEZE_STEPS, WARMUP_STEPS = 0, 250        # upstream: 1000 / 2000 (dành cho domain mới; domain 3 đã học sẵn)
COSINE, MIN_LR_RATIO = True, 0.1           # cosine về 0.1x tại step 5000 = endpoint chính
SEED = 42
MAX_HOURS = 10.5             # dừng có lưu checkpoint trước giới hạn 12h

XVLA_COMMIT = "6c4ecfb8be458ae7c0c96094e4b0add4174f4fe2"
CKPT_REPO, CKPT_REV = "2toINF/X-VLA-Libero", "129e71460678b7236cee6fc9707f09d9fa0c3590"
INPUT, WORK, TMP = Path("/kaggle/input"), Path("/kaggle/working"), Path("/kaggle/tmp")
for d in (WORK, TMP): d.mkdir(parents=True, exist_ok=True)
DATA, RUNS = WORK / "xvla_v2", WORK / "runs"
print({p: round(shutil.disk_usage(p).free / 2**30, 1) for p in (str(WORK), str(TMP))}, "GiB free")

## 1. Cài đặt

In [ ]:
pkgs = ["transformers==4.51.3", "peft==0.17.1", "accelerate==1.2.1", "mmengine==0.10.5", "einops==0.8.1",
        "timm==1.0.12", "json-numpy==2.1.0", "safetensors==0.4.5", "av", "fastapi", "uvicorn", "h5py", "scipy"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Code, checkpoint, shard

In [ ]:
CODE = next((p.parent for p in INPUT.rglob("train_xvla_mem.py")), None)
assert CODE, "Chưa gắn dataset xvla-v2-code (thư mục chứa train_xvla_mem.py)"
CODE_DIR = WORK / "code" / "xvla_v2"
shutil.copytree(CODE, CODE_DIR, dirs_exist_ok=True)

XVLA = next((p.parent for p in INPUT.rglob("peft_train.py") if (p.parent / "datasets").is_dir()), None)
if XVLA is None:
    XVLA = WORK / "code" / "X-VLA"
    if not (XVLA / "peft_train.py").is_file():
        subprocess.run(["git", "clone", "-q", "https://github.com/2toinf/X-VLA.git", str(XVLA)], check=True)
    subprocess.run(["git", "-C", str(XVLA), "checkout", "-q", XVLA_COMMIT], check=True)
    head = subprocess.run(["git", "-C", str(XVLA), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == XVLA_COMMIT, head

MODEL = next((p.parent for p in INPUT.rglob("model.safetensors") if (p.parent / "processing_xvla.py").is_file()), None)
if MODEL is None:
    from huggingface_hub import snapshot_download
    MODEL = TMP / "x_vla_libero"
    snapshot_download(CKPT_REPO, revision=CKPT_REV, local_dir=str(MODEL))
SHARDS = next((m.parent for m in INPUT.rglob("manifest.json") if (m.parent / "shards").is_dir()), None)
if SHARDS is None:  # dataset được giữ ở dạng zip
    z = next(INPUT.rglob("libero_mem_t1_t3_shards_v2.zip"))
    SHARDS = TMP / "shards_v2"
    if not (SHARDS / "manifest.json").is_file():
        subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(SHARDS)], check=True)
    SHARDS = next(m.parent for m in SHARDS.rglob("manifest.json") if (m.parent / "shards").is_dir())
print("code", CODE_DIR, "\nX-VLA", XVLA, "\nmodel", MODEL, "\nshards", SHARDS)

## 2b. Khôi phục checkpoint từ output phiên trước (nếu có)

In [ ]:
for st in INPUT.rglob("state.json"):
    ck = st.parent
    if ck.name.startswith("ckpt-") and ck.parent.name in ("f0", "f1", "smoke_f1") and (ck / "optimizer.pt").is_file():
        dst = RUNS / ck.parent.name / ck.name
        if not dst.exists():
            shutil.copytree(ck, dst); print("restored", dst)
        log = ck.parent / "train_log.jsonl"
        if log.is_file() and not (dst.parent / "train_log.jsonl").exists(): shutil.copy2(log, dst.parent / "train_log.jsonl")

## 3. Dựng dữ liệu X-VLA (CPU, vài phút)
Ghi `xvla_abs_actions_v2.h5` (~15 MB), `metas_f0/`, `metas_f1/`, `build_receipt.json`. Script dừng nếu split lock, số episode
(400 base + 198 Mem), độ dài, cờ `has_memory` hoặc ngân sách 50 token Florence-2 không khớp.

In [ ]:
def run(cmd, **kw):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    for line in p.stdout: print(line, end="")
    if p.wait(): raise RuntimeError(f"exit {p.returncode}")

if not (DATA / "build_receipt.json").is_file():
    run([sys.executable, CODE_DIR / "build_xvla_v2_dataset.py", "--shards-root", SHARDS, "--out", DATA, "--tokenizer", MODEL])
rc = json.loads((DATA / "build_receipt.json").read_text())
print(json.dumps({k: rc[k] for k in ("train_episodes", "frames", "sampler_frames", "zero_rotation_rows", "token_check")}, indent=1))

## 4. Kiểm tra căn chỉnh với checkpoint gốc (GPU, ~5 phút) — bắt buộc trước khi train
Checkpoint X-VLA-Libero đã giải được LIBERO-10, nên trên frame base nó phải dự đoán gần target của ta.
`rotation_frame_ok` và `flip_ok` phải là `true`; nếu sai, **dừng** và báo lại, đừng train.

In [ ]:
run([sys.executable, CODE_DIR / "check_xvla_alignment.py", "--xvla-root", XVLA, "--models", MODEL,
     "--metas", DATA / "metas_f0", "--episodes", 20, "--per-episode", 6, "--out", DATA / "alignment_base.json"])
v = json.loads((DATA / "alignment_base.json").read_text())["verdict"]; print(v)
assert v["rotation_frame_ok"] and v["flip_ok"], "Căn chỉnh không đạt, không train"

## 5. Smoke 30 update (đo s/update, VRAM, kiểm resume)

In [ ]:
def train(arm, out, iters, save_every, extra=()):
    metas = DATA / ("metas_f0" if arm == "F0" else "metas_f1")
    cmd = [sys.executable, "-m", "accelerate.commands.launch", "--num_processes", GPUS, "--num_machines", 1,
           "--mixed_precision", "fp16", "--dynamo_backend", "no"] + (["--multi_gpu"] if GPUS > 1 else []) + [
           CODE_DIR / "train_xvla_mem.py", "--xvla-root", XVLA, "--models", MODEL, "--metas", metas,
           "--output-dir", out, "--batch-size", BATCH_PER_GPU, "--grad-accum", GRAD_ACCUM, "--iters", iters,
           "--save-interval", save_every, "--learning-rate", LR, "--learning-coef", LR_COEF,
           "--freeze-steps", FREEZE_STEPS, "--warmup-steps", WARMUP_STEPS, "--min-lr-ratio", MIN_LR_RATIO,
           "--use-cosine-decay" if COSINE else "--no-cosine-decay", "--seed", SEED, "--resume", "auto", *extra]
    run(cmd, env=dict(os.environ, TOKENIZERS_PARALLELISM="false"))

SMOKE = RUNS / "smoke_f1"
train("F1", SMOKE, 20, 20, ["--log-interval", 5])
train("F1", SMOKE, 30, 10, ["--log-interval", 5])   # phải in "resumed from .../ckpt-20"
for l in (SMOKE / "train_log.jsonl").read_text().splitlines()[-3:]: print(l)

**Ước lượng thời gian:** `sec_per_update × 5000 / 3600` giờ cho mỗi arm. Nếu `frac_has_memory` ở F1 ≈ 0 thì prompt Mem
không vào được batch, dừng lại kiểm tra. Ở F0 giá trị này phải đúng bằng 0.

## 6. Train F0 và F1 (tự resume, dừng an toàn theo MAX_HOURS)

In [ ]:
t_session = time.time()
for arm in ARMS:
    out = RUNS / arm.lower()
    left = MAX_HOURS - (time.time() - t_session) / 3600
    if left < 0.5: print("Hết giờ phiên này, chạy lại notebook để resume", arm); break
    train(arm, out, ITERS, SAVE_EVERY, ["--max-hours", f"{left:.2f}"])
    done = max([json.loads((c / "state.json").read_text())["global_step"] for c in out.glob("ckpt-*")] or [0])
    print(arm, "step", done, "/", ITERS)
    if done < ITERS: break   # giữ đúng thứ tự: xong F0 mới sang F1

## 7. Đóng gói adapter + manifest (tải về để chạy đánh giá vòng kín)

In [ ]:
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
for arm in ARMS:
    out = RUNS / arm.lower(); ck = out / f"ckpt-{ITERS}"
    if not ck.is_dir(): print(arm, "chưa xong"); continue
    man = dict(arm=arm, base_checkpoint=CKPT_REPO, base_revision=CKPT_REV, xvla_commit=XVLA_COMMIT,
               steps=ITERS, effective_batch=BATCH_PER_GPU * GRAD_ACCUM * GPUS, lr=LR, freeze_steps=FREEZE_STEPS,
               warmup_steps=WARMUP_STEPS, cosine=COSINE, min_lr_ratio=MIN_LR_RATIO, lr_coef=LR_COEF, seed=SEED,
               trainable_parameters=json.loads((ck / "state.json").read_text()).get("trainable_parameters"), split_lock_sha256=rc["split_lock_sha256"],
               actions_h5_sha256=rc["actions_h5_sha256"], memory_source=rc["memory_source"],
               prompt_mode_mem="memory" if arm == "F1" else "instruction",
               files={f.name: sha(f) for f in ck.iterdir() if f.is_file() and f.name != "optimizer.pt"})
    (ck / "export_manifest.json").write_text(json.dumps(man, indent=1))
    z = WORK / f"xvla_v2_{arm.lower()}_step{ITERS}"
    tmpd = TMP / z.name; shutil.rmtree(tmpd, ignore_errors=True); shutil.copytree(ck, tmpd, ignore=shutil.ignore_patterns("optimizer.pt"))
    for f in ("build_receipt.json", "alignment_base.json"): shutil.copy2(DATA / f, tmpd / f)
    shutil.copy2(out / "train_log.jsonl", tmpd / "train_log.jsonl")
    print(shutil.make_archive(str(z), "zip", tmpd))